## Data Objects Used in the Trading Pipeline

The main inputs are organised into three groups:

1. **Market data panels**: prices, public traded volume, and alpha signals.
2. **Model parameter tables**: scaling parameters and fitted impact parameters.
3. **Strategy objects**: alpha dictionaries, trade dictionaries, and final backtest outputs.

---

<table>
<tr>
<td width="50%">

### A. Panel DataFrames

| Object | Index | Columns | Values | Used for |
|---|---|---|---|---|
| `test_px_df` | `(stock, date)` | Intraday time bins | Mid prices | Backtesting and return calculation |
| `test_traded_volume_df` | `(stock, date)` | Intraday time bins | Public signed volume | Public impact adjustment |
| `synthetic_alpha_df` | `(stock, date)` | Intraday time bins | Intraday alpha | OW optimal strategy |
| `strategy_trades_df` | `(stock, date)` | Intraday time bins | Signed strategy trades | Backtest input |

All of these objects have the same basic panel structure:

```text
index   = MultiIndex(stock, date)
columns = intraday time bins
values  = price / volume / alpha / trade

### B. Parameter DataFrames

| Object | Index | Relevant columns | Meaning |
|---|---|---|---|
| `scaling_df` | `stock` | `ADV`, `sigma` | Stock-level liquidity and volatility scaling |
| `stock_results_df` | default integer index | all model / half-life fits | Raw impact model fitting output |
| `best_df` | default integer index | best half-life by model | Selects the best half-life for each impact model |
| `fit_df` | `stock` | `lambda_hat`, `half_life_seconds` | Fitted parameters for one chosen impact model |

`fit_df` is created from:

```text
impact_model_stock_results_201901_train_201902_test.csv
impact_model_best_by_is_201901_train_201902_test.csv

stock_results_df = all stock-level fits for all models and all half-lives
best_df          = selected best half-life for each model
fit_df           = stock-level fitted parameters for one model at its best half-life

## Dictionaries Created Before Running the Pipeline

Before passing the objects into the trading/backtesting pipeline, we organise them into three dictionaries:

1. `fit_dfs` — fitted impact model parameters  
2. `alpha_dfs` — alpha signal panels  
3. `strategy_trade_dfs` — strategy trade panels  

---

### 1. Fitted Impact Model Dictionary

```python
fit_dfs = {
    "ow": ow_fit_df,
    "afs": afs_fit_df,
    "reduced_form": reduced_fit_df,
}
```

| Key | DataFrame | Index | Main columns | Meaning |
|---|---|---|---|---|
| `"ow"` | `ow_fit_df` | `stock` | `lambda_hat`, `half_life_seconds` | Fitted OW impact parameters |
| `"afs"` | `afs_fit_df` | `stock` | `lambda_hat`, `half_life_seconds` | Fitted AFS-style impact parameters |
| `"reduced_form"` | `reduced_fit_df` | `stock` | `lambda_hat`, `half_life_seconds` | Fitted reduced-form impact parameters |

Each dataframe has the same structure:

```text
index   = stock
columns = lambda_hat, half_life_seconds, model_type, is_r2, oos_r2, ...
```

---

### 2. Alpha Signal Dictionary

```python
alpha_dfs = {
    "intraday_only": synthetic_alpha_df,
    "night_only": overnight_alpha_df,
    "combined": combined_alpha_df,
}
```

| Key | DataFrame | Index | Columns | Values |
|---|---|---|---|---|
| `"intraday_only"` | `synthetic_alpha_df` | `(stock, date)` | intraday time bins | intraday alpha |
| `"night_only"` | `overnight_alpha_df` | `(stock, date)` | intraday time bins | overnight alpha repeated through the day |
| `"combined"` | `combined_alpha_df` | `(stock, date)` | intraday time bins | intraday + overnight alpha |

Each alpha dataframe has the same panel structure:

```text
index   = MultiIndex(stock, date)
columns = intraday time bins
values  = alpha signal
```

The alpha signal itself is not model-specific. The same intraday alpha can be passed into different impact-model backtests. The model-specific distinction appears later in the trade/backtest dictionaries.

---

### 3. Strategy Trade Dictionaries

Because the project separates **daily-reset** and **rolling/carry** backtests, the strategy trades are organised into two dictionaries.

---

#### A. Non-Rolling / Daily-Reset Strategy Trades

```python
non_rolling_strategy_trade_dfs = {
    "non_rolling_ow_day_only": ow_day_only_trades_df,
    "non_rolling_afs_day_only": afs_day_only_trades_df,
    "non_rolling_reduced_day_only": reduced_day_only_trades_df,
}
```

| Key | DataFrame | Index | Columns | Values |
|---|---|---|---|---|
| `"non_rolling_ow_day_only"` | `ow_day_only_trades_df` | `(stock, date)` | intraday time bins | OW optimal day-only trades |
| `"non_rolling_afs_day_only"` | `afs_day_only_trades_df` | `(stock, date)` | intraday time bins | AFS optimal day-only trades |
| `"non_rolling_reduced_day_only"` | `reduced_day_only_trades_df` | `(stock, date)` | intraday time bins | Reduced-form optimal day-only trades |

These strategies reset position and impact state at the beginning of each day.

---

#### B. Rolling / Carry Strategy Trades

```python
rolling_strategy_trade_dfs = {
    "rolling_ow_day_only": rolling_ow_day_only_trades_df,
    "rolling_afs_day_only": rolling_afs_day_only_trades_df,
    "rolling_reduced_day_only": rolling_reduced_day_only_trades_df,
    "rolling_ow_night_only": rolling_ow_night_only_trades_df,
    "rolling_ow_combined": rolling_ow_combined_trades_df,
}
```

| Key | DataFrame | Index | Columns | Values |
|---|---|---|---|---|
| `"rolling_ow_day_only"` | `rolling_ow_day_only_trades_df` | `(stock, date)` | intraday time bins | Rolling OW day-only trades |
| `"rolling_afs_day_only"` | `rolling_afs_day_only_trades_df` | `(stock, date)` | intraday time bins | Rolling AFS day-only trades |
| `"rolling_reduced_day_only"` | `rolling_reduced_day_only_trades_df` | `(stock, date)` | intraday time bins | Rolling reduced-form day-only trades |
| `"rolling_ow_night_only"` | `rolling_ow_night_only_trades_df` | `(stock, date)` | intraday time bins | Rolling OW trades from overnight alpha |
| `"rolling_ow_combined"` | `rolling_ow_combined_trades_df` | `(stock, date)` | intraday time bins | Rolling OW trades from intraday + overnight alpha |

These strategies carry position and impact state across days.

---

Each trade dataframe has the same panel structure:

```text
index   = MultiIndex(stock, date)
columns = intraday time bins
values  = signed trade q
```

In total, this gives eight strategy variants:

| Group | Strategies |
|---|---|
| Non-rolling day-only | OW, AFS, reduced-form |
| Rolling day-only | OW, AFS, reduced-form |
| Rolling overnight extension | OW night-only, OW combined |

 ## 1. Imports and paths

In [12]:
import pandas as pd
import numpy as np

from src.optimal_trading_strategies import *
from src.backtest_engine import *
from src.overnight_carry_backtest import *

DATA_DIR = "data"

## 2. Load core market data

In [13]:
test_px_df = load_panel_csv(
    f"{DATA_DIR}/test_px_201902_20.csv"
)

test_traded_volume_df = load_panel_csv(
    f"{DATA_DIR}/test_traded_volume_201902_20.csv"
)

scaling_df = load_stock_level_csv(
    f"{DATA_DIR}/scaling_201901_20.csv"
)

## 3. Load fitted impact model outputs

In [14]:
# %%
stock_results_df = pd.read_csv(
    f"{DATA_DIR}/impact_model_stock_results_201901_train_201902_test.csv"
)

best_df = pd.read_csv(
    f"{DATA_DIR}/impact_model_best_by_is_201901_train_201902_test.csv"
)

ow_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="ow",
)

afs_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="afs",
)

reduced_fit_df = get_best_model_fit_df(
    stock_results_df=stock_results_df,
    best_df=best_df,
    model_type="reduced_form",
)

## 4. Create fitted impact model dictionary

In [15]:
fit_dfs = {
    "ow": ow_fit_df,
    "afs": afs_fit_df,
    "reduced_form": reduced_fit_df,
}

## 5. Load and create alpha panels

In [8]:
synthetic_alpha_df = load_panel_csv(
    f"{DATA_DIR}/synthetic_alpha_201902_20.csv"
)

In [ ]:
# EXAMINE BETTER WHAT THIS FUNCTION IS CREATING

overnight_alpha_df, overnight_return_s = make_next_open_overnight_alpha_panel(
    test_px_df=test_px_df,
    overnight_alpha_level=1.0,
)

combined_alpha_df = make_combined_alpha_df(
    intraday_alpha_df=synthetic_alpha_df,
    overnight_alpha_df=overnight_alpha_df,
    intraday_weight=1.0,
    overnight_weight=1.0,
)

## 6. Create alpha dictionary

In [10]:
alpha_dfs = {
    "intraday_only": synthetic_alpha_df,
    "night_only": overnight_alpha_df,
    "combined": combined_alpha_df,
}

## 7. Load non-rolling / daily-reset strategy trades

In [ ]:
# CHECK NAMES BETTER (WRONG NAME)
ow_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_non_rolling_ow_day_only_201902_20.csv"
)

# STILL NOT AVAILABLE
afs_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_non_rolling_afs_day_only_201902_20.csv"
)

reduced_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_non_rolling_reduced_day_only_201902_20.csv"
)

## 8. Create non-rolling strategy trade dictionary

In [ ]:
non_rolling_strategy_trade_dfs = {
    "non_rolling_ow_day_only": ow_day_only_trades_df,
    "non_rolling_afs_day_only": afs_day_only_trades_df,
    "non_rolling_reduced_day_only": reduced_day_only_trades_df,
}

## 9. Load rolling / carry strategy trades

In [ ]:
rolling_ow_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_rolling_ow_day_only_201902_20.csv"
)

rolling_afs_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_rolling_afs_day_only_201902_20.csv"
)

rolling_reduced_day_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_rolling_reduced_day_only_201902_20.csv"
)

rolling_ow_night_only_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_rolling_ow_night_only_201902_20.csv"
)

rolling_ow_combined_trades_df = load_panel_csv(
    f"{DATA_DIR}/strategy_trades_rolling_ow_combined_201902_20.csv"
)

## 10. Create rolling strategy trade dictionary

In [ ]:
rolling_strategy_trade_dfs = {
    "rolling_ow_day_only": rolling_ow_day_only_trades_df,
    "rolling_afs_day_only": rolling_afs_day_only_trades_df,
    "rolling_reduced_day_only": rolling_reduced_day_only_trades_df,
    "rolling_ow_night_only": rolling_ow_night_only_trades_df,
    "rolling_ow_combined": rolling_ow_combined_trades_df,
}

## 11. Final object check

In [ ]:
print("Core data:")
print("test_px_df:", test_px_df.shape)
print("test_traded_volume_df:", test_traded_volume_df.shape)
print("scaling_df:", scaling_df.shape)

print("\nFit dictionaries:")
for name, df in fit_dfs.items():
    print(name, df.shape)

print("\nAlpha dictionaries:")
for name, df in alpha_dfs.items():
    print(name, df.shape)

print("\nNon-rolling strategy trades:")
for name, df in non_rolling_strategy_trade_dfs.items():
    print(name, df.shape)

print("\nRolling strategy trades:")
for name, df in rolling_strategy_trade_dfs.items():
    print(name, df.shape)

## PLOTS

In [ ]:
stock = "AAPL"
date = "2019-02-21"

paths, summaries = run_one_stock_day_paths_for_strategies(
    stock=stock,
    date=date,
    strategy_trade_dfs={
        "OW optimal": ow_day_only_trades_df,
        "AFS optimal": afs_day_only_trades_df,
        "Reduced-form optimal": reduced_day_only_trades_df,
    },
    model_type="reduced_form",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=10,
)

summaries

In [ ]:
plot_one_stock_day_pnl_comparison(paths, stock, date)

In [ ]:
plot_one_stock_day_final_pnl_bar(summaries, stock, date)

In [ ]:
plot_one_stock_day_impact_cost_bar(summaries, stock, date)

In [ ]:
model_comparison_df = run_strategy_backtests_for_model(
    strategy_trade_dfs=non_rolling_strategy_trade_dfs,
    model_type="reduced_form",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=DT_SECONDS,
)

display(model_comparison_df[
    (model_comparison_df["stock"] == "AAPL")
    & (model_comparison_df["date"].astype(str) == "2019-02-21")
][[
    "strategy",
    "daily_pnl",
    "impact_cost",
    "max_abs_impact",
    "final_position",
    "total_abs_traded",
    "net_traded",
    "model_type",
    "stock",
    "date",
]])

In [ ]:
summary = (
    model_comparison_df
    .groupby("strategy")
    .agg(
        mean_pnl_bps=("pnl_bps", "mean"),
        median_pnl_bps=("pnl_bps", "median"),
        std_pnl_bps=("pnl_bps", "std"),
        mean_impact_cost_bps=("impact_cost_bps", "mean"),
        mean_max_abs_impact=("max_abs_impact", "mean"),
        mean_abs_traded=("total_abs_traded", "mean"),
    )
)

summary["daily_sharpe"] = summary["mean_pnl_bps"] / summary["std_pnl_bps"]

display(summary.round(4))

# 2.6 Performance metrics

In [ ]:
alpha_corr_intraday = alpha_forward_return_corr(
    alpha_df=alpha_dfs["intraday_only"],
    px_df=test_px_df,
    horizon_bins=1,
)

alpha_corr_intraday

In [ ]:
alpha_corr_5min = alpha_forward_return_corr(
    alpha_df=alpha_dfs["intraday_only"],
    px_df=test_px_df,
    horizon_bins=30,   # 5 minutes = 30 ten-second bins
)
alpha_corr_5min

In [ ]:
strategy_trade_dfs = {
    "ow_optimal": ow_day_only_trades_df,
    "afs_optimal": afs_day_only_trades_df,
    "reduced_form_optimal": reduced_day_only_trades_df,
}

ow_sim_results = run_strategy_backtests_for_model(
    strategy_trade_dfs=strategy_trade_dfs,
    model_type="ow",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=DT_SECONDS,
)

afs_sim_results = run_strategy_backtests_for_model(
    strategy_trade_dfs=strategy_trade_dfs,
    model_type="afs",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=DT_SECONDS,
)

reduced_sim_results = run_strategy_backtests_for_model(
    strategy_trade_dfs=strategy_trade_dfs,
    model_type="reduced_form",
    test_px_df=test_px_df,
    test_traded_volume_df=test_traded_volume_df,
    scaling_df=scaling_df,
    fit_dfs=fit_dfs,
    dt_seconds=DT_SECONDS,
)

all_model_comparison_df = pd.concat(
    [ow_sim_results, afs_sim_results, reduced_sim_results],
    ignore_index=True,
)

In [ ]:
all_model_comparison_df.head()

In [ ]:
daily_portfolio_df = build_daily_portfolio_df(all_model_comparison_df)


performance_report_df = summarize_strategy_performance(daily_portfolio_df)
performance_report_df["alpha_1bin_corr"] = alpha_corr_intraday
performance_report_df["alpha_5min_corr"] = alpha_corr_5min
display(performance_report_df.round(4))

In [ ]:
# Sensitivity
compact_report_df = performance_report_df[[
    "backtest_model",
    "strategy",
    "expected_daily_pnl_bps",
    "daily_sharpe",
    "annualized_sharpe",
    "mean_transaction_cost_bps",
    "max_daily_drawdown_bps",
    "max_impact_dislocation",
    "n_days",
]]

display(compact_report_df.round(4))

In [ ]:
for model in daily_portfolio_df["backtest_model"].unique():
    plot_cumulative_pnl_by_model(daily_portfolio_df, model)

In [ ]:
# Compact version 
own_model_results_df = all_model_comparison_df[
    (
        (all_model_comparison_df["strategy"] == "ow_optimal")
        & (all_model_comparison_df["backtest_model"] == "ow")
    )
    | (
        (all_model_comparison_df["strategy"] == "afs_optimal")
        & (all_model_comparison_df["backtest_model"] == "afs")
    )
    | (
        (all_model_comparison_df["strategy"] == "reduced_form_optimal")
        & (all_model_comparison_df["backtest_model"] == "reduced_form")
    )
].copy()

daily_portfolio_df = build_daily_portfolio_df(own_model_results_df)
performance_report_df = summarize_strategy_performance(daily_portfolio_df)

performance_report_df["alpha_1bin_corr"] = alpha_corr_1bin
performance_report_df["alpha_5min_corr"] = alpha_corr_5min

display(performance_report_df.round(4))